In [2]:
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset,DataLoader


import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report


In [3]:
df=pd.read_csv('cleaned_data.csv')
df.head()

,label,Text
0,0,meeting ã¼ rite ill go home lor ã¼ dun feel li...
1,0,ill always even spirit ill get bb soon trying ...
2,0,sorry took long omw
3,0,thk 50 shd ok said plus minus 10 ã¼ leave line...
4,0,dunno juz askin cos got card got 20 4 salon ca...


In [4]:
df['Text']=df['Text'].fillna("").astype(str)
df=df.dropna(subset=['label'])

In [5]:
X=df['Text']
y=df['label'].astype(int)

In [6]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)

In [7]:
vectorizer=TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

In [8]:
X_train_vec=vectorizer.fit_transform(X_train)
X_test_vec=vectorizer.transform(X_test)

In [9]:
X_train_np=X_train_vec.toarray().astype(np.float32)
X_test_np=X_test_vec.toarray().astype(np.float32)

y_train_np=y_train.to_numpy().astype(np.float32)
y_test_np=y_test.to_numpy().astype(np.float32)

In [10]:
def evaluate_model(y_true, probabilities, threshold=0.5):
    predictions = (np.asarray(probabilities).ravel() >= threshold).astype(int)

    print("Accuracy :", accuracy_score(y_true, predictions))
    print("Precision:", precision_score(y_true, predictions, zero_division=0))
    print("Recall   :", recall_score(y_true, predictions, zero_division=0))
    print("F1-score :", f1_score(y_true, predictions, zero_division=0))
    print("\nClassification Report:\n")
    print(classification_report(y_true, predictions, zero_division=0))


In [11]:
torch.manual_seed(42)

# Convert data into tensors

In [12]:
X_train_t=torch.tensor(X_train_np,dtype=torch.float32)
X_test_t=torch.tensor(X_test_np,dtype=torch.float32)

y_train_t=torch.tensor(
    y_train_np.reshape(-1,1),dtype=torch.float32
)

In [14]:
dataset=TensorDataset(
    X_train_t,y_train_t
)

In [18]:
loader=DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

In [21]:
class SpamClassifier(nn.Module):
    def __init__(self,input_features):
        super().__init__()
        self.network=nn.Sequential(
            nn.Linear(input_features,128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )
    def forward(self,x):
        return self.network(x)

In [22]:
model=SpamClassifier(
    X_train_np.shape[1]
)

In [24]:
criterion=nn.BCEWithLogitsLoss()
optimizer=torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs=10


for epoch in range(epochs):
    model.train()
    total_loss=0.0

    for batch_X,batch_y in loader:
        logits=model(batch_X)
        loss=criterion(logits,batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss+=loss.item()*len(batch_X)
    avg_loss=total_loss/len(dataset)
    print(f"Epoch {epoch+1}/{epochs}, Loss {avg_loss:.4f}")


Epoch 1/10, Loss 0.6746
Epoch 2/10, Loss 0.4246
Epoch 3/10, Loss 0.0980
Epoch 4/10, Loss 0.0354
Epoch 5/10, Loss 0.0181
Epoch 6/10, Loss 0.0141
Epoch 7/10, Loss 0.0087
Epoch 8/10, Loss 0.0087
Epoch 9/10, Loss 0.0091
Epoch 10/10, Loss 0.0069


In [25]:
model.eval()
with torch.no_grad():
    logits=model(X_test_t)
    y_prob=torch.sigmoid(logits).cpu().numpy().ravel()


evaluate_model(y_test_np, y_prob)

Accuracy : 0.9464882943143813
Precision: 0.9463087248322147
Recall   : 0.9463087248322147
F1-score : 0.9463087248322147

Classification Report:

              precision    recall  f1-score   support

         0.0       0.95      0.95      0.95       150
         1.0       0.95      0.95      0.95       149

    accuracy                           0.95       299
   macro avg       0.95      0.95      0.95       299
weighted avg       0.95      0.95      0.95       299



In [26]:
# Predict a new SMS
message = ["Congratulations! You have won a free prize"]
message_vec = vectorizer.transform(message).toarray().astype(np.float32)
message_t = torch.tensor(message_vec, dtype=torch.float32)

model.eval()
with torch.no_grad():
    logits = model(message_t)
    probability = torch.sigmoid(logits).item()

print("Spam" if probability >= 0.5 else "Not Spam")
print("Spam probability:", round(probability, 4))


Spam
Spam probability: 0.997
